# Backtest ATR Trend Envelope - EURUSD
Este notebook realiza a simulação do Expert Advisor `envelope1.mq5` para o par EURUSD (Proxy: EURUSD=X) utilizando dados do Yahoo Finance.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Download de Dados Históricos
Baixando dados de 1 hora para o período solicitado.

In [ ]:
data = yf.download('EURUSD=X', start='2025-03-14', end='2026-03-16', interval='1h')
df = data.copy()
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
df.head()

## 2. Implementação do Indicador ATR Trend Envelope
Replicando a lógica de cálculo manual do True Range e as bandas de suporte/resistência do MQL5.

In [ ]:
inpAtrPeriod = 14
inpDeviation = 1.5

df['TR'] = 0.0
for i in range(1, len(df)):
    df.loc[df.index[i], 'TR'] = max(df.iloc[i]['High'], df.iloc[i-1]['Close']) - min(df.iloc[i]['Low'], df.iloc[i-1]['Close'])

df['ATR'] = df['TR'].rolling(window=inpAtrPeriod).mean()

s_min = np.zeros(len(df))
s_max = np.zeros(len(df))
trend = np.zeros(len(df))

for i in range(1, len(df)):
    atr = df.iloc[i]['ATR']
    if pd.isna(atr): continue
        
    dev = atr * inpDeviation
    val_h, val_l, val_c = df.iloc[i]['High'], df.iloc[i]['Low'], df.iloc[i]['Close']
    
    s_max[i] = val_h + dev
    s_min[i] = val_l - dev
    
    prev_trend, prev_s_max, prev_s_min = trend[i-1], s_max[i-1], s_min[i-1]
    
    if val_c > prev_s_max and prev_s_max > 0: trend[i] = 1
    elif val_c < prev_s_min and prev_s_min > 0: trend[i] = -1
    else: trend[i] = prev_trend
        
    if trend[i] > 0 and s_min[i] < prev_s_min: s_min[i] = prev_s_min
    if trend[i] < 0 and s_max[i] > prev_s_max: s_max[i] = prev_s_max

df['Smin'], df['Smax'], df['Trend'] = s_min, s_max, trend

## 3. Simulação da Estratégia
Executando ordens em mudanças de tendência com Stop Loss estrutural e realizações parciais.

In [ ]:
balance = 100000.0
risk_percent = 0.01
rr_ratio = 2.5
be_trigger_atr = 1.0
trailing_multiplier = 1.5
contract_size = 100000

positions = []
history = []

for i in range(2, len(df)):
    if df.iloc[i]['Trend'] != df.iloc[i-1]['Trend'] and df.iloc[i-1]['Trend'] != 0:
        for p in positions:
            p['exit_price'] = df.iloc[i]['Open']
            history.append((p['exit_price'] - p['entry_price']) * p['direction'] * p['lot_size'] * contract_size)
        positions = []
        
        direction = df.iloc[i]['Trend']
        entry_price = df.iloc[i]['Open']
        sl = df.iloc[i-1]['Smax'] if direction == 1 else df.iloc[i-1]['Smin']
        
        if sl == 0: continue
        dist = abs(entry_price - sl)
        if dist == 0: continue
        
        tp = entry_price + dist * rr_ratio * direction
        lot_size = (balance * risk_percent) / (dist * contract_size)
        
        positions.append({
            'entry_price': entry_price, 'sl': sl, 'tp': tp, 'direction': direction,
            'lot_size': lot_size, 'initial_vol': lot_size, 'current_vol': lot_size,
            'be_protected': False, 'p25': False, 'p50': False, 'p75': False, 'status': 'open'
        })

    high, low, close, atr = df.iloc[i]['High'], df.iloc[i]['Low'], df.iloc[i]['Close'], df.iloc[i]['ATR']
    for p in positions[:]:
        if (p['direction'] == 1 and low <= p['sl']) or (p['direction'] == -1 and high >= p['sl']):
            history.append((p['sl'] - p['entry_price']) * p['direction'] * p['current_vol'] * contract_size)
            positions.remove(p)
        elif (p['direction'] == 1 and high >= p['tp']) or (p['direction'] == -1 and low <= p['tp']):
            history.append((p['tp'] - p['entry_price']) * p['direction'] * p['current_vol'] * contract_size)
            positions.remove(p)
        else:
            prog = ((close - p['entry_price']) * p['direction']) / abs(p['tp'] - p['entry_price'])
            if prog >= 0.25 and not p['p25']:
                history.append((close - p['entry_price']) * p['direction'] * (p['initial_vol']*0.25) * contract_size)
                p['current_vol'] -= (p['initial_vol']*0.25); p['p25'] = True; p['sl'] = p['entry_price']
            elif prog >= 0.50 and not p['p50']:
                history.append((close - p['entry_price']) * p['direction'] * (p['initial_vol']*0.25) * contract_size)
                p['current_vol'] -= (p['initial_vol']*0.25); p['p50'] = True
            elif prog >= 0.75 and not p['p75']:
                history.append((close - p['entry_price']) * p['direction'] * (p['initial_vol']*0.25) * contract_size)
                p['current_vol'] -= (p['initial_vol']*0.25); p['p75'] = True
            
            if not p['be_protected'] and (close - p['entry_price']) * p['direction'] >= atr * be_trigger_atr:
                p['sl'] = p['entry_price']; p['be_protected'] = True
            
            new_sl = close - atr * trailing_multiplier if p['direction'] == 1 else close + atr * trailing_multiplier
            if (p['direction'] == 1 and new_sl > p['sl']) or (p['direction'] == -1 and (p['sl'] == 0 or new_sl < p['sl'])): p['sl'] = new_sl

## 4. Resultados e Métricas

In [ ]:
history = np.array(history)
net_profit = np.sum(history)
profit_factor = abs(np.sum(history[history > 0]) / np.sum(history[history <= 0]))
sharpe = (np.mean(history) / np.std(history)) * np.sqrt(252)
payoff = net_profit / len(history)

cumulative_profit = np.cumsum(history)
balance_evolution = 100000 + cumulative_profit
peak, max_dd = 100000, 0
for val in balance_evolution:
    if val > peak: peak = val
    dd = (peak - val) / 100000 * 100
    if dd > max_dd: max_dd = dd

print(f"Lucro Líquido Total: {net_profit:.2f} USD")
print(f"Fator de Lucro: {profit_factor:.2f}")
print(f"Índice Sharpe: {sharpe:.2f}")
print(f"Retorno Esperado (Payoff): {payoff:.2f} USD")
print(f"Drawdown Máximo: {max_dd:.2f}%")

plt.figure(figsize=(12, 6))
plt.plot(balance_evolution, label='Saldo da Conta (Equity)', color='blue')
plt.axhline(y=100000, color='red', linestyle='--', label='Saldo Inicial')
plt.title('Evolução do Saldo - EURUSD H1')
plt.legend(); plt.grid(True)
plt.show()